In [2]:
import requests
import time

# ================================================================
# ENDPOINTS
# ================================================================
QWEN3_ENDPOINT = "http://ec2-98-81-228-187.compute-1.amazonaws.com:11434"
GEMMA3_ENDPOINT = "http://ec2-100-31-82-64.compute-1.amazonaws.com:11434"
EMBEDDINGS_ENDPOINT = "http://ec2-3-208-23-94.compute-1.amazonaws.com:11434"


# ================================================================
# HELPERS
# ================================================================
def query_model(endpoint, model, prompt):
    """Query a chat model and return response + time"""
    try:
        start = time.time()
        response = requests.post(
            f"{endpoint}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=120,
        )
        elapsed = time.time() - start
        result = response.json()
        return result.get("response", "No response"), elapsed, None
    except Exception as e:
        return None, 0, str(e)


def get_embeddings(endpoint, model, text):
    """Get embeddings for a text and return vector + time"""
    try:
        start = time.time()
        response = requests.post(
            f"{endpoint}/api/embeddings",
            json={"model": model, "prompt": text},
            timeout=30,
        )
        elapsed = time.time() - start
        result = response.json()
        embedding = result.get("embedding", [])
        return embedding, elapsed, None
    except Exception as e:
        return None, 0, str(e)


def check_health(endpoint, model):
    """Check if a model is available"""
    try:
        response = requests.get(f"{endpoint}/api/tags", timeout=10)
        models = response.json().get("models", [])
        available = [m["name"] for m in models]
        return model in available or any(model in m for m in available)
    except Exception:
        return False


# ================================================================
# TESTS
# ================================================================
def test_health_checks():
    print("\n" + "=" * 60)
    print("🔍 HEALTH CHECKS")
    print("=" * 60)

    checks = [
        (QWEN3_ENDPOINT, "qwen3:8b"),
        (GEMMA3_ENDPOINT, "gemma3:12b"),
        (EMBEDDINGS_ENDPOINT, "mxbai-embed-large"),
    ]

    for endpoint, model in checks:
        status = check_health(endpoint, model)
        icon = "✅" if status else "❌"
        print(f"{icon} {model} — {endpoint}")


def test_chat_models():
    print("\n" + "=" * 60)
    print("🤖 CHAT MODELS TEST")
    print("=" * 60)

    prompt = "In one sentence, what is fleet telemetry in mining operations?"

    models = [
        (QWEN3_ENDPOINT, "qwen3:8b", "Qwen3 8B"),
        (GEMMA3_ENDPOINT, "gemma3:12b", "Gemma3 12B"),
    ]

    for endpoint, model, name in models:
        print(f"\n📤 Prompt: {prompt}")
        print(f"🤖 Model: {name}")
        response, elapsed, error = query_model(endpoint, model, prompt)
        if error:
            print(f"❌ Error: {error}")
        else:
            print(f"📥 Response: {response}")
            print(f"⏱️  Time: {elapsed:.2f}s")


def test_embeddings():
    print("\n" + "=" * 60)
    print("🔢 EMBEDDINGS TEST")
    print("=" * 60)

    texts = [
        "Mining fleet telemetry analysis",
        "Fuel consumption anomaly detection",
        "Fatigue monitoring for truck drivers",
    ]

    for text in texts:
        print(f"\n📤 Text: {text}")
        embedding, elapsed, error = get_embeddings(
            EMBEDDINGS_ENDPOINT, "mxbai-embed-large", text
        )
        if error:
            print(f"❌ Error: {error}")
        else:
            print(f"📐 Dimensions: {len(embedding)}")
            print(f"🔢 First 5 values: {embedding[:5]}")
            print(f"⏱️  Time: {elapsed:.2f}s")


def test_similarity():
    print("\n" + "=" * 60)
    print("🔍 SIMILARITY TEST")
    print("=" * 60)

    import math

    def cosine_similarity(v1, v2):
        dot = sum(a * b for a, b in zip(v1, v2))
        mag1 = math.sqrt(sum(a**2 for a in v1))
        mag2 = math.sqrt(sum(b**2 for b in v2))
        return dot / (mag1 * mag2) if mag1 and mag2 else 0

    texts = [
        "Truck fuel consumption is abnormally high",
        "Vehicle is using too much fuel",
        "The weather is sunny today",
    ]

    embeddings = []
    for text in texts:
        emb, _, error = get_embeddings(EMBEDDINGS_ENDPOINT, "mxbai-embed-large", text)
        if not error:
            embeddings.append((text, emb))

    if len(embeddings) >= 2:
        base_text, base_emb = embeddings[0]
        print(f"\n📌 Base: '{base_text}'")
        for text, emb in embeddings[1:]:
            similarity = cosine_similarity(base_emb, emb)
            print(f"🔗 vs '{text}' → similarity: {similarity:.4f}")


# ================================================================
# MAIN
# ================================================================
if __name__ == "__main__":
    print("\n🚀 MineLogX AI - Model Tests")
    print("=" * 60)

    test_health_checks()
    test_embeddings()
    test_similarity()
    test_chat_models()

    print("\n" + "=" * 60)
    print("✅ Tests completed!")
    print("=" * 60)


🚀 MineLogX AI - Model Tests

🔍 HEALTH CHECKS
✅ qwen3:8b — http://ec2-98-81-228-187.compute-1.amazonaws.com:11434
✅ gemma3:12b — http://ec2-100-31-82-64.compute-1.amazonaws.com:11434
✅ mxbai-embed-large — http://ec2-3-208-23-94.compute-1.amazonaws.com:11434

🔢 EMBEDDINGS TEST

📤 Text: Mining fleet telemetry analysis
📐 Dimensions: 1024
🔢 First 5 values: [0.023310218006372452, 0.04794234409928322, -0.004706582985818386, 0.0071024405770003796, -0.005584041588008404]
⏱️  Time: 0.31s

📤 Text: Fuel consumption anomaly detection
📐 Dimensions: 1024
🔢 First 5 values: [-0.009185326285660267, 0.02732216753065586, 0.014793450944125652, -0.005967167671769857, -0.013841561041772366]
⏱️  Time: 0.26s

📤 Text: Fatigue monitoring for truck drivers
📐 Dimensions: 1024
🔢 First 5 values: [0.05246545001864433, 0.04116589576005936, -0.017668433487415314, 0.033814936876297, -0.0372379794716835]
⏱️  Time: 0.25s

🔍 SIMILARITY TEST

📌 Base: 'Truck fuel consumption is abnormally high'
🔗 vs 'Vehicle is using too mu

In [10]:
# ============================================================
# INSPECT S3 VECTORS INDEX — run this at any time to check
# the current state of the index independently of the pipeline
# ============================================================

SOURCE_BUCKET = "bhitech-minelogx-poc-legislation-documents"
VECTOR_BUCKET = "bhitech-minelogx-poc-legislation-documents-vector"
INDEX_NAME = "bhitech-minelogx-chunkspdfs"
PDF_PREFIX = "docs"  # root folder to scan
FOLDER_LIMIT = None  # set to an int (e.g. 10) to process a subset only
AWS_REGION = "us-east-1"

import boto3
from collections import defaultdict

# Reuse the constants already declared in the notebook
# (SOURCE_BUCKET, VECTOR_BUCKET, INDEX_NAME, AWS_REGION)

_s3v = boto3.client("s3vectors", region_name=AWS_REGION)
_s3 = boto3.client("s3", region_name=AWS_REGION)

# ----------------------------------------------------------
# 1. Count total vectors currently stored in the index
# ----------------------------------------------------------
total_vectors = 0
next_token = None

while True:
    params = {
        "vectorBucketName": VECTOR_BUCKET,
        "indexName": INDEX_NAME,
        "maxResults": 100,
        "returnMetadata": False,  # faster — we only need the count
    }
    if next_token:
        params["nextToken"] = next_token

    response = _s3v.list_vectors(**params)
    batch = response.get("vectors", [])
    total_vectors += len(batch)
    next_token = response.get("nextToken")

    if not next_token:
        break

print(f"Total vectors in index : {total_vectors:,}")

# ----------------------------------------------------------
# 2. Count PDFs available in the source bucket
# ----------------------------------------------------------
total_pdfs = 0
paginator = _s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=SOURCE_BUCKET, Prefix=PDF_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].lower().endswith(".pdf"):
            total_pdfs += 1

print(f"Total PDFs in source   : {total_pdfs:,}")

# ----------------------------------------------------------
# 3. Breakdown: vectors per source PDF
#    (requires returnMetadata=True — slower but informative)
# ----------------------------------------------------------
print("\nVectors per source document:")
print("-" * 60)

vectors_by_source = defaultdict(int)
next_token = None

while True:
    params = {
        "vectorBucketName": VECTOR_BUCKET,
        "indexName": INDEX_NAME,
        "maxResults": 100,
        "returnMetadata": True,
    }
    if next_token:
        params["nextToken"] = next_token

    response = _s3v.list_vectors(**params)

    for v in response.get("vectors", []):
        source_key = v.get("metadata", {}).get("source_key", "unknown")
        # Show only the filename, not the full S3 path
        filename = source_key.split("/")[-1] if source_key != "unknown" else "unknown"
        vectors_by_source[filename] += 1

    next_token = response.get("nextToken")
    if not next_token:
        break

if vectors_by_source:
    for filename, count in sorted(vectors_by_source.items()):
        print(f"  {count:>4} vectors  —  {filename}")
else:
    print("  (index is empty)")

print("-" * 60)
print(f"  {total_vectors:>4} vectors total across {len(vectors_by_source)} document(s)")

Total vectors in index : 788
Total PDFs in source   : 7

Vectors per source document:
------------------------------------------------------------
   788 vectors  —  CELEX_32014L0034_EN_TXT.pdf
------------------------------------------------------------
   788 vectors total across 1 document(s)


In [5]:
response = requests.get(
    "http://ec2-3-208-23-94.compute-1.amazonaws.com:11434/api/tags", timeout=10
)
print(response.json())

{'models': [{'name': 'mxbai-embed-large:latest', 'model': 'mxbai-embed-large:latest', 'modified_at': '2026-06-03T19:46:54.567941867Z', 'size': 669615493, 'digest': '468836162de7f81e041c43663fedbbba921dcea9b9fefea135685a39b2d83dd8', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'bert', 'families': ['bert'], 'parameter_size': '334M', 'quantization_level': 'F16', 'context_length': 512, 'embedding_length': 1024}, 'capabilities': ['embedding']}]}
